In [1]:
import os, json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

load_dotenv()

/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
with open("bajaj_db.json") as f:
    db = json.load(f)
    

In [ ]:
# step 1: define your tool( your python functions)


@tool
def get_loan_status(loan_id: str) -> dict:
    """Fetches current status of a Bajaj Finance loan from the database.
    Returns EMI amount, remaining months, outstanding balance, next due date.
    Use this when customer asks about their loan details, EMI, or balance.
    Args:
        loan_id: The loan account number (e.g., 'BFL2024001')
    """
    if loan_id not in db["loans"]:
        return {"error": f"Loan {loan_id} not found"}
    loan = db["loans"][loan_id]
    return {
        "customer_name": loan["customer_name"],
        "emi": loan["emi"],
        "remaining_months": loan["remaining_months"],
        "outstanding": loan["outstanding"],
        "next_due_date": loan["next_due_date"],
        "prepayment_charge_pct": loan["prepayment_charge_pct"]
    }

@tool
def sayhello():
    """This function will welcome to my customers. Call this fucntion when user say hello
    """
    return {
        "user_message": " Hello, Today I am ooo"
    }


ValueError: Function must have a docstring if description not provided.

In [24]:
# step 2 : bind your list of tools to your llm
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

tools = [get_loan_status,sayhello]
llm_with_tools = llm.bind_tools(tools)


In [27]:
llm_with_tools.invoke('hello')

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 131, 'total_tokens': 141, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0628d073e2', 'id': 'chatcmpl-Day0kjJyt1qnuOq7tYY7GaIRCI2rJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019de758-fb43-7aa2-adc2-18b560159d88-0', tool_calls=[{'name': 'sayhello', 'args': {}, 'id': 'call_DD1QjhZAqXeF6gxzb9UO4p1a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 131, 'output_tokens': 10, 'total_tokens': 141, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [30]:
# Customer message
system = SystemMessage(content="You are a Bajaj Finance support agent. Use tools for real data.")
user_msg = HumanMessage(content="hello")

message = [system,user_msg]
response = llm_with_tools.invoke(message)

In [31]:
response.tool_calls

[{'name': 'sayhello',
  'args': {},
  'id': 'call_zA1O8X5WdIVUthhizAeb9kfD',
  'type': 'tool_call'}]

In [32]:
tool_map={"get_loan_status":get_loan_status,"sayhello":sayhello
}

tool_result = []

for tc in response.tool_calls:
    tool_name = tc["name"]
    print("tool name: ",tool_name)
    tool_args = tc["args"]
    print("tool_args ",tool_args)
    tool_id = tc["id"]

    # call your tool --> execute ??
    result = tool_map[tool_name].invoke(tool_args)
    # print('result from tools',result)
    tool_result.append((tool_id,tool_name,result))

tool name:  sayhello
tool_args  {}


In [33]:
tool_result

[('call_zA1O8X5WdIVUthhizAeb9kfD',
  'sayhello',
  {'user_message': ' Hello, Today I am ooo'})]

In [34]:
# step 4: pass the result to llm with Tool message --> get the final user ou

message = [system,user_msg,response]

for tool_id,tool_name, result in tool_result:
    tool_msg = ToolMessage(
        content = str(result) ,# must be in string
        tool_call_id = tool_id

    )

    message.append(tool_msg)



In [35]:
message

[SystemMessage(content='You are a Bajaj Finance support agent. Use tools for real data.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hello', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 146, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9075db19fa', 'id': 'chatcmpl-Day12p0B8mo5SYHIfcliVku3Ypy0v', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019de759-41d9-7803-b538-d40eef8827ec-0', tool_calls=[{'name': 'sayhello', 'args': {}, 'id': 'call_zA1O8X5WdIVUthhizAeb9kfD', 'type': 'tool_call'}], invalid_tool_calls=[], usage

In [36]:
# step 5 : pass entire messages --> llm

final_response = llm_with_tools.invoke(message)

In [37]:
final_response

AIMessage(content='Hello! Today I am here to assist you. How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 177, 'total_tokens': 195, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9075db19fa', 'id': 'chatcmpl-Day1q6gcDZgbmnH8BfGNl558dN5qr', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019de75a-04fb-72d2-8ee3-847adceed76b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 177, 'output_tokens': 18, 'total_tokens': 195, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})